# PolypDB -> YOLOv8 Segmentation (Colab)

This notebook prepares **PolypDB modality-wise data** for **YOLOv8 segmentation**, trains the model, computes mask-based metrics comparable with U-Net, and downloads the final metrics files.

In [ ]:

# Install dependencies
!pip install -q ultralytics kagglehub opencv-python pandas openpyxl pyyaml


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 51.0 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:

import os
import cv2
import json
import math
import shutil
import random
import zipfile
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict

import kagglehub
from ultralytics import YOLO
from google.colab import files


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:


# 1. Download PolypDB

path = kagglehub.dataset_download("debeshjha1/polypdb")
print("Path to dataset files:", path)

ROOT = os.path.join(path, "PolypDB", "PolypDB")
print("ROOT:", ROOT)


100%|██████████| 1.29G/1.29G [00:33<00:00, 41.4MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/debeshjha1/polypdb/versions/1
ROOT: /root/.cache/kagglehub/datasets/debeshjha1/polypdb/versions/1/PolypDB/PolypDB


In [ ]:


# 2. Select modality

# Options: "BLI", "FICE", "LCI", "NBI", "WLI"
MODALITY = "WLI"

IMG_DIR  = os.path.join(ROOT, "PolypDB_modality_wise", MODALITY, "images")
MASK_DIR = os.path.join(ROOT, "PolypDB_modality_wise", MODALITY, "masks")

assert os.path.exists(IMG_DIR), f"Image folder not found: {IMG_DIR}"
assert os.path.exists(MASK_DIR), f"Mask folder not found: {MASK_DIR}"

print("IMG_DIR :", IMG_DIR)
print("MASK_DIR:", MASK_DIR)
print("Images  :", len(os.listdir(IMG_DIR)))
print("Masks   :", len(os.listdir(MASK_DIR)))


IMG_DIR : /root/.cache/kagglehub/datasets/debeshjha1/polypdb/versions/1/PolypDB/PolypDB/PolypDB_modality_wise/WLI/images
MASK_DIR: /root/.cache/kagglehub/datasets/debeshjha1/polypdb/versions/1/PolypDB/PolypDB/PolypDB_modality_wise/WLI/masks
Images  : 3588
Masks   : 3588


In [ ]:

# 3. Build YOLOv8-seg dataset


OUT_ROOT = f"/content/polypdb_yolo_{MODALITY.lower()}"
TRAIN_RATIO = 0.80
SEED = 42
MIN_POINTS = 6

random.seed(SEED)
np.random.seed(SEED)

for sub in [
    "images/train", "images/val",
    "labels/train", "labels/val",
    "masks/train",  "masks/val"
]:
    os.makedirs(os.path.join(OUT_ROOT, sub), exist_ok=True)


def sorted_basenames(img_dir, mask_dir):
    img_map = {Path(f).stem: f for f in os.listdir(img_dir) if not f.startswith('.')}
    mask_map = {Path(f).stem: f for f in os.listdir(mask_dir) if not f.startswith('.')}
    common = sorted(set(img_map) & set(mask_map))
    return common, img_map, mask_map


def mask_to_yolo_segments(mask_path):
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return []

    h, w = mask.shape
    binary = (mask > 127).astype(np.uint8) * 255
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    segments = []
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < 10:
            continue
        cnt = cnt.squeeze(1)
        if cnt.ndim != 2 or len(cnt) < 3:
            continue

        coords = []
        for x, y in cnt:
            coords.extend([x / w, y / h])

        if len(coords) >= MIN_POINTS:
            segments.append(coords)

    return segments


basenames, img_map, mask_map = sorted_basenames(IMG_DIR, MASK_DIR)
print("Matched pairs:", len(basenames))

random.shuffle(basenames)
cut = int(len(basenames) * TRAIN_RATIO)
train_ids = set(basenames[:cut])
val_ids   = set(basenames[cut:])

for stem in basenames:
    split = "train" if stem in train_ids else "val"

    src_img  = os.path.join(IMG_DIR, img_map[stem])
    src_mask = os.path.join(MASK_DIR, mask_map[stem])

    dst_img  = os.path.join(OUT_ROOT, f"images/{split}", os.path.basename(src_img))
    dst_mask = os.path.join(OUT_ROOT, f"masks/{split}", os.path.basename(src_mask))
    dst_lab  = os.path.join(OUT_ROOT, f"labels/{split}", f"{stem}.txt")

    shutil.copy2(src_img, dst_img)
    shutil.copy2(src_mask, dst_mask)

    segments = mask_to_yolo_segments(src_mask)
    with open(dst_lab, "w") as f:
        for seg in segments:
            f.write("0 " + " ".join(f"{v:.6f}" for v in seg) + "\n")

print("Train images:", len(os.listdir(os.path.join(OUT_ROOT, "images/train"))))
print("Val images  :", len(os.listdir(os.path.join(OUT_ROOT, "images/val"))))


Matched pairs: 3588
Train images: 2870
Val images  : 718


In [ ]:


# 4. Dataset YAML

YAML_PATH = os.path.join(OUT_ROOT, "polypdb.yaml")

yaml_text = f"""
path: {OUT_ROOT}
train: images/train
val: images/val

names:
  0: polyp
""".strip()

with open(YAML_PATH, "w") as f:
    f.write(yaml_text)

print(open(YAML_PATH).read())


path: /content/polypdb_yolo_wli
train: images/train
val: images/val

names:
  0: polyp


In [ ]:


# 5. Train YOLOv8 segmentation

MODEL_NAME = "yolov8n-seg.pt"   # change to yolov8s-seg.pt if you want a larger model
EPOCHS = 100
IMGSZ = 352
BATCH = 8
PATIENCE = 25
PROJECT = "/content/runs_polypdb_yolo"
RUN_NAME = f"yolov8_seg_{MODALITY.lower()}"

model = YOLO(MODEL_NAME)

train_results = model.train(
    data=YAML_PATH,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=PATIENCE,
    project=PROJECT,
    name=RUN_NAME,
    pretrained=True,
    seed=SEED,
    save=True,
    verbose=True
)

BEST_MODEL = os.path.join(PROJECT, RUN_NAME, "weights", "best.pt")
print("Best model:", BEST_MODEL)


Ultralytics 8.4.36 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition, 97250MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/polypdb_yolo_wli/polypdb.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=352, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8_seg_wli, nbs=64, nms=False, opset=None, optimize=False, op

In [ ]:

# 6. Official Ultralytics validation

best_model = YOLO(BEST_MODEL)
val_results = best_model.val(
    data=YAML_PATH,
    split="val",
    imgsz=IMGSZ,
    batch=BATCH,
    plots=True,
    save_json=False,
    verbose=True
)
print(val_results)


Ultralytics 8.4.36 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition, 97250MiB)
YOLOv8n-seg summary (fused): 86 layers, 3,258,259 parameters, 0 gradients, 11.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 7913.9±2032.0 MB/s, size: 155.5 KB)
val: Scanning /content/polypdb_yolo_wli/labels/val.cache... 718 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 718/718 376.4Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 90/90 57.9it/s 1.6s
                   all        718        778      0.929      0.865      0.944      0.782      0.917        0.9      0.949      0.755
Speed: 0.1ms preprocess, 0.5ms inference, 0.0ms loss, 0.4ms postprocess per image
Results saved to /content/runs/segment/val
ultralytics.utils.metrics.SegmentMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object


In [ ]:


# 7. Mask metrics comparable to U-Net

import torch
@torch.no_grad()
def compute_binary_metrics(gt_mask, pred_mask, eps=1e-7, beta=2.0):
    gt = (gt_mask > 0).astype(np.uint8).reshape(-1)
    pr = (pred_mask > 0).astype(np.uint8).reshape(-1)

    tp = np.sum((gt == 1) & (pr == 1))
    fp = np.sum((gt == 0) & (pr == 1))
    fn = np.sum((gt == 1) & (pr == 0))

    iou = (tp + eps) / (tp + fp + fn + eps)
    dice = (2 * tp + eps) / (2 * tp + fp + fn + eps)
    precision = (tp + eps) / (tp + fp + eps)
    recall = (tp + eps) / (tp + fn + eps)

    beta2 = beta ** 2
    f2 = ((1 + beta2) * precision * recall + eps) / (beta2 * precision + recall + eps)
    return iou, dice, recall, precision, f2


def predict_mask(model, image_path):
    results = model.predict(source=image_path, imgsz=IMGSZ, conf=0.25, verbose=False)
    r = results[0]
    h, w = r.orig_shape
    pred = np.zeros((h, w), dtype=np.uint8)

    if r.masks is not None and len(r.masks.data) > 0:
        masks = r.masks.data.cpu().numpy()
        merged = (np.max(masks, axis=0) > 0.5).astype(np.uint8)
        if merged.shape != (h, w):
            merged = cv2.resize(merged, (w, h), interpolation=cv2.INTER_NEAREST)
        pred = merged

    return pred


In [ ]:
# Missing import used above
import torch

# 8. Evaluate all validation images

VAL_IMG_DIR  = os.path.join(OUT_ROOT, "images", "val")
VAL_MASK_DIR = os.path.join(OUT_ROOT, "masks",  "val")

rows = []

for img_name in sorted(os.listdir(VAL_IMG_DIR)):
    if img_name.startswith('.'):
        continue

    stem = Path(img_name).stem
    img_path = os.path.join(VAL_IMG_DIR, img_name)

    # find GT mask with same stem
    gt_candidates = [f for f in os.listdir(VAL_MASK_DIR) if Path(f).stem == stem]
    if not gt_candidates:
        print(f"Warning: no GT mask for {stem}")
        continue

    gt_path = os.path.join(VAL_MASK_DIR, gt_candidates[0])
    gt_mask = cv2.imread(gt_path, cv2.IMREAD_GRAYSCALE)
    gt_mask = (gt_mask > 127).astype(np.uint8)

    pred_mask = predict_mask(best_model, img_path)
    iou, dice, recall, precision, f2 = compute_binary_metrics(gt_mask, pred_mask)

    rows.append({
        "id": stem,
        "IoU": float(iou),
        "DSC": float(dice),
        "Recall": float(recall),
        "Precision": float(precision),
        "F2": float(f2)
    })

per_image_df = pd.DataFrame(rows)
print("Validation images evaluated:", len(per_image_df))
per_image_df.head()


Validation images evaluated: 718


,id,IoU,DSC,Recall,Precision,F2
0,00314c36-1a16-4682-84a1-6d6ad1211a7b,0.879415,0.935839,0.964099,0.909189,0.952592
1,003ab8b5-cf6d-442a-849d-80ddf0d1edaf,0.947740,0.973169,0.985635,0.961014,0.980610
2,004d98d2-d6c8-4c09-9b4f-fe13d6c6bdcd,0.790677,0.883104,0.982383,0.802050,0.940108
3,007bfb9d-5903-4109-895b-d5e5dfe91f9a,0.911515,0.953709,0.916125,0.994509,0.930798
4,00be7d4a-d643-4075-9ea3-e5bff2cd9a5c,0.960478,0.979841,0.983045,0.976658,0.981761


In [ ]:


# 9. Aggregate table for paper comparison

summary = {
    "Method": "YOLOv8-seg",
    "Backbone": MODEL_NAME.replace('.pt', ''),
    "Dataset": f"PolypDB ({MODALITY})",
    "mIoU": float(per_image_df["IoU"].mean()),
    "mDSC": float(per_image_df["DSC"].mean()),
    "Recall": float(per_image_df["Recall"].mean()),
    "Precision": float(per_image_df["Precision"].mean()),
    "F2": float(per_image_df["F2"].mean())
}

summary_df = pd.DataFrame([summary])
summary_df


,Method,Backbone,Dataset,mIoU,mDSC,Recall,Precision,F2
0,YOLOv8-seg,yolov8n-seg,PolypDB (WLI),0.787457,0.862098,0.894743,0.871378,0.878129


In [ ]:


# 10. Save all metrics files

per_image_csv = f"yolov8_{MODALITY.lower()}_per_image_metrics.csv"
summary_csv   = f"yolov8_{MODALITY.lower()}_summary_metrics.csv"
excel_file    = f"yolov8_{MODALITY.lower()}_metrics.xlsx"
json_file     = f"yolov8_{MODALITY.lower()}_metrics.json"
zip_file      = f"yolov8_{MODALITY.lower()}_results_bundle.zip"

per_image_df.to_csv(per_image_csv, index=False)
summary_df.to_csv(summary_csv, index=False)

with pd.ExcelWriter(excel_file, engine="openpyxl") as writer:
    summary_df.to_excel(writer, sheet_name="summary", index=False)
    per_image_df.to_excel(writer, sheet_name="per_image", index=False)

payload = {
    "summary": summary,
    "per_image_count": int(len(per_image_df)),
    "best_model_path": BEST_MODEL
}
with open(json_file, "w") as f:
    json.dump(payload, f, indent=2)

# Also copy official training CSV if it exists
official_csv = os.path.join(PROJECT, RUN_NAME, "results.csv")
if os.path.exists(official_csv):
    shutil.copy2(official_csv, f"yolov8_{MODALITY.lower()}_official_train_results.csv")

# zip bundle
files_to_zip = [per_image_csv, summary_csv, excel_file, json_file, BEST_MODEL]
extra = f"yolov8_{MODALITY.lower()}_official_train_results.csv"
if os.path.exists(extra):
    files_to_zip.append(extra)

with zipfile.ZipFile(zip_file, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in files_to_zip:
        if os.path.exists(f):
            zf.write(f, arcname=os.path.basename(f))

print("Saved:")
for f in [per_image_csv, summary_csv, excel_file, json_file, zip_file]:
    print(" -", f, os.path.exists(f))


Saved:
 - yolov8_wli_per_image_metrics.csv True
 - yolov8_wli_summary_metrics.csv True
 - yolov8_wli_metrics.xlsx True
 - yolov8_wli_metrics.json True
 - yolov8_wli_results_bundle.zip True


In [ ]:


# 11. Download final files

for fname in [
    f"yolov8_{MODALITY.lower()}_per_image_metrics.csv",
    f"yolov8_{MODALITY.lower()}_summary_metrics.csv",
    f"yolov8_{MODALITY.lower()}_metrics.xlsx",
    f"yolov8_{MODALITY.lower()}_metrics.json",
    f"yolov8_{MODALITY.lower()}_results_bundle.zip",
]:
    if os.path.exists(fname):
        files.download(fname)
    else:
        print("Missing:", fname)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os
import shutil

DRIVE_PATH = "/content/drive/MyDrive/yolo_metrics"
os.makedirs(DRIVE_PATH, exist_ok=True)


# 12. Save final files to Google Drive

for fname in [
    f"yolov8_{MODALITY.lower()}_per_image_metrics.csv",
    f"yolov8_{MODALITY.lower()}_summary_metrics.csv",
    f"yolov8_{MODALITY.lower()}_metrics.xlsx",
    f"yolov8_{MODALITY.lower()}_metrics.json",
    f"yolov8_{MODALITY.lower()}_results_bundle.zip",
]:
    if os.path.exists(fname):
        dst = os.path.join(DRIVE_PATH, fname)
        shutil.copy(fname, dst)
        print(f"Saved to Drive: {dst}")
    else:
        print("Missing:", fname)

Saved to Drive: /content/drive/MyDrive/yolo_metrics/yolov8_wli_per_image_metrics.csv
Saved to Drive: /content/drive/MyDrive/yolo_metrics/yolov8_wli_summary_metrics.csv
Saved to Drive: /content/drive/MyDrive/yolo_metrics/yolov8_wli_metrics.xlsx
Saved to Drive: /content/drive/MyDrive/yolo_metrics/yolov8_wli_metrics.json
Saved to Drive: /content/drive/MyDrive/yolo_metrics/yolov8_wli_results_bundle.zip
